In [11]:
import numpy as np
import pandas as pd

from subprocess import check_output
print(check_output('dir .\\input', shell=True).decode('cp949'))

 C 드라이브의 볼륨에는 이름이 없습니다.
 볼륨 일련 번호: 2602-0473

 c:\Users\USER\Desktop\AI_study\kaggle_transcription\Binary classification - Image classification\1st level. Statoil C-CORE Iceberg Classifier Challenge\input 디렉터리

2026-04-06  오후 06:53    <DIR>          .
2026-04-07  오후 01:20    <DIR>          ..
2026-04-06  오후 06:53    <DIR>          data
2026-03-07  오후 08:56            38,566 sample_submission.csv.7z
2026-03-07  오후 08:56       257,127,394 test.json.7z
2026-03-07  오후 08:56        44,932,785 train.json.7z
               3개 파일         302,098,745 바이트
               3개 디렉터리  1,484,321,660,928 바이트 남음



This kernel is specifically is for Beginners who wants to experiment building CNN using Keras. By using this kernel, you can expect to get good score and also learn keras. Keras is simple frameworks where we can initialize the model and keep stacking the layers we want. It makes building deep neural networks very easy.

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from os.path import join as opj
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pylab
plt.rcParams['figure.figsize'] = 10, 10
%matplotlib inline

In [15]:
train = pd.read_json('./input/train.json')

In [16]:
test = pd.read_json('./input/test.json')

# Intro about the Data.

Sentinet -1 sat is at about 680 Km above earth. Sending pulses of signals at a particular angle of incidence and then recording it back. Basically, those reflected signals are called backscatter. The data we have been given is backscatter coefficient which is the conventional form of backscatter coefficient given by:

$$\sigma{o}(dB) = \beta{o}(dB) + 10\log{10}\left[\frac{\sin(i_{p})}{\sin(i_{c})}\right]$$

where

1. ip=is angle of incidence for a particular pixel
2. 'ic' is angle of incidence or center of the image
3. K=constant

We have been given $\sigma{o}$ directly in the data.

### Now coming to the features of $\sigma{o}$

Basically $\sigma{o}$ varies with the surface on which the signal is scattered from. For example, for a particular angle of incidence, it varies like:

* WATER...........SETTLEMENTS........AGRICULTURE.........BARREN.........

1. HH: -27.001....2.70252................-12.7952..............-17.25790909
2. HV: -28.035.....-20.2665..............-21.4471..............-20.019

As you can see, the HH component varies a lot but HV doesn't. **I don't have the data for scatter from ship, but being a metal object, it should vary differently as compared to ice object**.

### WTF is HH HV?
Ok, so this Sentinel Satellite is equivalent to RISTSAT(an Indian remote sensing Sat) and they only Transmit pings in H polarization, **AND NOT IN V polarization**. Those H-pings gets scattered, objects change their polarization and returns as a mix of H and V. **Since Sentinel has only H-transmitter, return signals are of the form of HH and HV only**. Don't ask why VV is not given(because Sentinel don't have V-ping transmitter).

Now coming to features, for the purpose of this demo code, I am extracting all two bands and taking avg of them as 3rd channel to create a 3-channel RGB equivalent.

In [17]:
X_band_1 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in train['band_1']])
X_band_2 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in train['band_2']])
X_train = np.concatenate([X_band_1[:, :, :, np.newaxis], X_band_2[:, :, :, np.newaxis], ((X_band_1+X_band_2)/2)[:, :, :, np.newaxis]], axis=-1)

In [18]:
import plotly.offline as py
import plotly.graph_objs as go
py.init_notebook_mode(connected=True)
def plotmy3d(c, name):

    data = [
        go.Surface(
            z=c
        )
    ]
    layout = go.Layout(
        title=name,
        autosize=False,
        width=700,
        height=700,
        margin=dict(
            l=65,
            r=50,
            b=65,
            t=90
        )
    )
    fig = go.Figure(data=data, layout=layout)
    py.iplot(fig)
plotmy3d(X_band_1[12,:,:], 'iceberg')

That's a cool looking iceberg we have. Remember, in radar data, the shape of the iceberg is going to be like a mountain as shown in here. Since this is not a actual image but scatter from radar, the shape is going to have peaks and distortions like these. The shape of the ship is going to be like a point, may be like a elongated point. From here the structural differences arise and we can exploit those differences using a CNN. It would be helpful if we can create composite images using the backscatter from radar.

In [19]:
plotmy3d(X_band_1[14,:,:], 'Ship')

That's a ship, looks like a elongated point. We don't have much resolution in images to visualize the shape of the ship. However CNN is here to help. There are few papers on ship iceberg classification like this: http://elib.dir.de/99079/2/2016 BENETES Frost Velotto Tings EUSAR FP.pdf However their data have much better resolution so I don't feel that the CNN they used would be suitable here.

Get back to building a CNN using Keras. Much better frameworks then others. You will enjoy for sure.

In [24]:
from matplotlib import pyplot
from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.models import Sequential
from tf_keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Input, Flatten, Activation
from tf_keras.layers import GlobalMaxPooling2D
from tf_keras.layers import BatchNormalization
from tf_keras.layers import Concatenate
from tf_keras.models import Model
from tf_keras import initializers
from tf_keras.optimizers import Adam
from tf_keras.callbacks import ModelCheckpoint, Callback, EarlyStopping

In [28]:
def getModel():
    gmodel = Sequential()

    gmodel.add(Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=(75, 75, 3)))
    gmodel.add(MaxPooling2D(pool_size=(3, 3), strides=(2, 2)))
    gmodel.add(Dropout(0.2))

    gmodel.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
    gmodel.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    gmodel.add(Dropout(0.2))

    gmodel.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
    gmodel.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    gmodel.add(Dropout(0.2))

    gmodel.add(Flatten())

    gmodel.add(Dense(512))
    gmodel.add(Activation('relu'))
    gmodel.add(Dropout(0.2))

    gmodel.add(Dense(256))
    gmodel.add(Activation('relu'))
    gmodel.add(Dropout(0.2))

    gmodel.add(Dense(1))
    gmodel.add(Activation('sigmoid'))

    myoptim = Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
                #    decay=0.0
                   )
    gmodel.compile(loss='binary_crossentropy', optimizer=myoptim, metrics=['accuracy'])
    gmodel.summary()
    return gmodel

def get_callbacks(filepath, patience=2):
    es = EarlyStopping('val_loss', patience=patience, mode='min')
    msave = ModelCheckpoint(filepath, save_best_only=True)
    return [es, msave]

file_path = '.model_weights.hdf5'
callbacks = get_callbacks(filepath=file_path, patience=5)

In [29]:
target_train = train['is_iceberg']
X_train_cv, X_valid, y_train_cv, y_valid = train_test_split(X_train, target_train, random_state=1, train_size=0.75)

In [30]:
import os

gmodel = getModel()
gmodel.fit(X_train_cv, y_train_cv,
           batch_size=24,
           epochs=50,
           verbose=1,
           validation_data=(X_valid, y_valid),
           callbacks=callbacks)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 73, 73, 64)        1792      
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 dropout_5 (Dropout)         (None, 36, 36, 64)        0         
                                                                 
 conv2d_4 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 17, 17, 128)       0         
 g2D)                                                            
                                                                 
 dropout_6 (Dropout)         (None, 17, 17, 128)      

51/51 [==============================] - 8s 129ms/step - loss: 1.6080 - accuracy: 0.4904 - val_loss: 0.5878 - val_accuracy: 0.6633
Epoch 2/50
 1/51 [..............................] - ETA: 4s - loss: 0.6029 - accuracy: 0.7917

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning:

You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.



51/51 [==============================] - 5s 92ms/step - loss: 0.5915 - accuracy: 0.6376 - val_loss: 0.5792 - val_accuracy: 0.6559
Epoch 3/50
51/51 [==============================] - 5s 89ms/step - loss: 0.5692 - accuracy: 0.6600 - val_loss: 0.5883 - val_accuracy: 0.6608
Epoch 4/50
51/51 [==============================] - 5s 88ms/step - loss: 0.5521 - accuracy: 0.6758 - val_loss: 0.6119 - val_accuracy: 0.6983
Epoch 5/50
51/51 [==============================] - 5s 89ms/step - loss: 0.5650 - accuracy: 0.6592 - val_loss: 0.5685 - val_accuracy: 0.6683
Epoch 6/50
51/51 [==============================] - 5s 96ms/step - loss: 0.5385 - accuracy: 0.6983 - val_loss: 0.5519 - val_accuracy: 0.6883
Epoch 7/50
51/51 [==============================] - 5s 102ms/step - loss: 0.5228 - accuracy: 0.6991 - val_loss: 0.5299 - val_accuracy: 0.7157
Epoch 8/50
51/51 [==============================] - 5s 100ms/step - loss: 0.5189 - accuracy: 0.7132 - val_loss: 0.5264 - val_accuracy: 0.7506
Epoch 9/50
51/51 [====

**Through the score may be different here, it works good on LB, I got 0.210 score.**

In [31]:
gmodel.load_weights(filepath=file_path)
score = gmodel.evaluate(X_valid, y_valid, verbose=1)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 27ms/step - loss: 0.3917 - accuracy: 0.8229
Test loss: 0.39174190163612366
Test accuracy: 0.8229426145553589


In [34]:
X_band_test_1 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in test['band_1']])
X_band_test_2 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in test['band_2']])
X_test = np.concatenate([X_band_test_1[:, :, :, np.newaxis],
                         X_band_test_2[:, :, :, np.newaxis],
                         ((X_band_test_1+X_band_test_2)/2)[:, :, :, np.newaxis]], axis=-1)
predicted_test = gmodel.predict(X_test)

264/264 [==============================] - 7s 26ms/step


In [35]:
submission = pd.DataFrame()
submission['id'] = test['id']
submission['is_iceberg'] = predicted_test.reshape((predicted_test.shape[0]))
submission.to_csv('sub.csv', index=False)

### Conclusion

To increase the score, I have tried Speckle filtering, indicence angle normalization and other preprocessing and they don't seem to work. You may try and see but for me they are not giving any good results.

You can't be on top-10 using this kernel, so here is one beautiful piece of information. The test dataset contain 8000 images, we can exploit this. We can do pseudo labelling to increase the predictions. Here is the article related to that: https://towardsdatascience.com/simple-explanation-of-semi-supervised-learning-and-pseudo-labeling-c2218e8c769b

Upvote if you liked this kernel.